# Кибериммунный подход к разработке. Учебный пример "Светофор"

## Об авторе 

Этот блокнот разработан для вас Сергеем Соболевым, sergey.p.sobolev@kaspersky.com

Больше информации о кибериммунном подходе можно найти на странице https://github.com/sergey-sobolev/cyberimmune-systems/wiki/%D0%9A%D0%B8%D0%B1%D0%B5%D1%80%D0%B8%D0%BC%D0%BC%D1%83%D0%BD%D0%B8%D1%82%D0%B5%D1%82

Подписывайтесь на телеграм-канал @learning_cyberimmunity (https://t.me/learning_cyberimmunity)

Обучающие видео на тему кибериммунного подхода вы можете найти на youtube канале https://www.youtube.com/@learning_cyberimmunity/

## О примере 

Светофор - это, на первый взгляд, очень простая система, но она оказывает критическое влияние на безопасность дорожного движения. 
Применим идеи конструктивной безопасности к архитектуре и реализации прототипа светофора. За отправную точку возьмём [код базового примера](https://colab.research.google.com/github/sergey-sobolev/cyberimmune-systems-basic-demo-notebook01/blob/main/cyberimmunity-basics.ipynb) и немного его переработаем.

Для переработки будем использовать описание архитектуры решения, которое обсуждалось на занятии.

## Простая реализация выбранной политики архитектуры

Будем использовать политику архитектуры, показанную на рис. 1.

![Рис. 1. Политика архитектуры](images/tl-archpol-0.02.png)

Рис. 1. Политика архитектуры светофора

1. Создадим функциональные компоненты (сущности 1-4) и монитор безопасности, который будет контролировать их взаимодействие, в том числе реализовывать контроль конфигураций светофора (сущность №5 на архитектурной диаграмме)
2. Определим политики безопасности
3. Сымитируем запрос на изменение режима для проверки работы всех элементов

- В качестве интерфейса взаимодействия используем очереди сообщений, у каждой сущности есть своя «персональная» очередь, ассоциированная с ней
- Компоненты 1-4 отправляют сообщения только в очередь monitor сущности SecurityMonitor
- SecurityMonitor проверяет сообщения на соответствие политикам безопасности, в случае положительного решения перенаправляет сообщение в очередь соответствующей сущности

В коде назовём сущности следующим образом
1. Связь - CitySystemConnector
2. Система управления светофора - ControlSystem
3. Управление светодиодами - LightsGPIO
4. Система диагностики - SelfDiagnosticsSystem

Логику контроля режимов светофора (компонент №5 на рис. 1) реализуем в виде политик безопасности в мониторе безопасности

![Рис. 2. Политика архитектуры с именами классов](images/tl-archpol-code.png)

Рис. 2. Политика архитектуры с именами классов

Очередь событий для монитора безопасности: все запросы от сущностей друг к другу должны отправляться только в неё

In [12]:
from threading import Thread as Process
from queue import Queue, Empty
from dataclasses import dataclass
import json
from time import sleep
import time

monitor_events_queue = Queue()

Зафиксируем формат сообщений

In [13]:
from dataclasses import dataclass


@dataclass
class Event:
    source: str       
    destination: str  
    operation: str    
    parameters: str 

In [14]:
duration = 30

### Монитор безопасности

Ниже в методе _check_policies можно увидеть пример политики безопасности:

```python
if event.source == "ControlSystem" \
        and event.destination == "LightsGPIO" \
        and event.operation == "set_mode" \ 
        and self._check_mode(event.operation):
    authorized = True
```            

в этом примере проверяется отправитель сообщения, получатель, запрашиваемая операция и даже параметры операции. Это максимально жёсткий вариант, очевидно, в зависимости от ситуации количество проверок можно уменьшить.

А пока это место для экспериментов, как можно из монитора безопасности заблокировать взаимодействие между сущностями.

In [22]:

# формат управляющих команд для монитора
@dataclass
class ControlEvent:
    operation: str

# список разрешенных сочетаний сигналов светофора
# любые сочетания, отсутствующие в этом списке, запрещены
traffic_lights_allowed_configurations = [
    {"direction_1": "red", "direction_2": "green"},
    {"direction_1": "red", "direction_2": "red"},    
    {"direction_1": "red", "direction_2": "yellow"},    
    {"direction_1": "yellow", "direction_2": "yellow"},    
    {"direction_1": "off", "direction_2": "off"},
    {"direction_1": "green", "direction_2": "red"},    
    {"direction_1": "green", "direction_2": "yellow"},
    {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking"},
    {"direction_1": "left", "direction_2": "off"},
    {"direction_1": "right", "direction_2": "off"},    
    {"direction_1": "left_front", "direction_2": "off"},
     {"direction_1": "left_front", "direction_2": "right_front"},
    {"direction_1": "right_front", "direction_2": "off"},    
    {"direction_1": "off", "direction_2": "left"},
    {"direction_1": "off", "direction_2": "right"},
    {"direction_1": "green", "direction_2": "left"},    
    {"direction_1": "green", "direction_2": "right"},
    {"direction_1": "yellow", "direction_2": "left"},    
    {"direction_1": "yellow", "direction_2": "right"},
    {"direction_1": "red", "direction_2": "left"},    
    {"direction_1": "red", "direction_2": "right"},
    {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking"},
]


class Monitor(Process):
    def __init__(self, events_q: Queue, duration = duration):
        super().__init__()
        self._events_q = events_q 
        self._control_q = Queue() 
        self._entity_queues = {}   
        self._force_quit = False   
        self.daemon = True # Важно: чтобы поток умирал вместе с ядром
        self.duration = duration

    def add_entity_queue(self, entity_id: str, queue: Queue):
        print(f"[монитор] регистрируем сущность {entity_id}")
        self._entity_queues[entity_id] = queue

    def _check_mode(self, mode_dict: dict) -> bool:
        data = mode_dict
        if isinstance(mode_dict, str):
            try:
                data = json.loads(mode_dict)
            except:
                return False
        
        print(f"[монитор] deep inspection payload: {data}")
        return data in traffic_lights_allowed_configurations      

    def _check_policies(self, event):
        print(f'[монитор] обрабатываем событие {event}')
        if not isinstance(event, Event): return False

        authorized = False
        if event.source == "ControlSystem" \
                and event.destination == "LightsGPIO" \
                and event.operation == "set_mode" \
                and self._check_mode(event.parameters):
            authorized = True
            
        elif event.source == "LightsGPIO" \
                and event.destination == "SelfDiagnostics" \
                and event.operation == "diagnostic_log":
            authorized = True

        elif event.source == "CitySystemConnector" \
                and event.destination == "ControlSystem" \
                and (event.operation == "update_config" or event.operation == "force_mode"):
            authorized = True
            
        if authorized is False:
            print("[монитор] событие не разрешено политиками безопасности")
        return authorized

    def _proceed(self, event):
        print(f'[монитор] отправляем запрос {event}')
        try:
            dst_q: Queue = self._entity_queues[event.destination]
            dst_q.put(event)
        except Exception as e:
            print(f"[монитор] ошибка выполнения запроса {e}")
  
    def run(self):
        print('[монитор] старт')
        start_time = time.time()
        while time.time() - start_time < self.duration:
            try:
                event = self._events_q.get_nowait()
                if self._check_policies(event):
                    self._proceed(event)
            except Empty:
                sleep(0.1) # Чуть уменьшил задержку для скорости реакции
            except Exception as e:
                print(f"[монитор] ошибка обработки {e}")
            self._check_control_q()
        print('[монитор] завершение работы')

    def stop(self):
        request = ControlEvent(operation='stop')
        self._control_q.put(request)

    def _check_control_q(self):
        try:
            request = self._control_q.get_nowait()
            if isinstance(request, ControlEvent) and request.operation == 'stop':
                self._force_quit = True
        except Empty:
            pass

### Сущность ControlSystem

Эта сущность отправляет сообщение для другой сущности (LightsGPIO)

In [23]:
class ControlSystem(Process):
    def __init__(self, monitor_queue: Queue, duration = 40, green_duration = 15, yellow_duration = 5):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.daemon = True
        self.duration = duration
        self.green_duration = green_duration
        self.yellow_duration = yellow_duration
    def entity_queue(self): return self._own_queue

    def run(self):        
        print(f'[{self.__class__.__name__}] старт')
        start_time = time.time()
        sleep(1) 
        traffic_cycle = [
            ({"direction_1": "green", "direction_2": "red"}, self.green_duration),
            ({"direction_1": "yellow", "direction_2": "red"}, self.yellow_duration),
            ({"direction_1": "red", "direction_2": "green"}, self.green_duration),
            ({"direction_1": "red", "direction_2": "yellow"}, self.yellow_duration),
        ]

        cycle_index = 0

        while time.time() - start_time < self.duration:
            current_mode, sleep_time = traffic_cycle[cycle_index]
            self._send_mode(current_mode)
            spent = 0
            
            while spent < sleep_time:
                if time.time() - start_time >= self.duration:
                    break
                sleep(0.1)
                spent += 0.1

            cycle_index = (cycle_index + 1) % len(traffic_cycle)

        print(f'[{self.__class__.__name__}] завершение работы')

### Сущность LightsGPIO

Эта сущность ждёт входящее сообщение в течение заданного периода времени, если получает - обрабатывает и завершает работу или выходит по таймауту.

In [32]:
class LightsGPIO(Process):
    def __init__(self, monitor_queue: Queue, duration = duration):
        super().__init__()
        self.monitor_queue = monitor_queue
        self._own_queue = Queue()
        self.daemon = True
        self.duration = duration
        
    def entity_queue(self): return self._own_queue

    def run(self):  
        start_time = time.time()
        print(f'[{self.__class__.__name__}] старт')
        while time.time() - start_time < self.duration:
            try:
                
                event = self._own_queue.get_nowait()
                if event.operation == "set_mode":
                    print(f"[{self.__class__.__name__}] !!! ПОЛУЧЕНО !!! {event.parameters}")
                    self._send_diag(f"Режим сменен на {event.parameters}")
                    log = Event("LightsGPIO", "SelfDiagnostics", "log", f"Режим сменен: {event.parameters}")
                    self.monitor_queue.put(log)
                    break
                    
                elif event.operation == "update_config":
                    cfg = json.loads(event.parameters)
                        if "green_duration" in cfg:
                            print(f"[ControlSystem] Конфигурация обновлена: Длительность зеленого = {cfg['green_duration']}")
                            self.green_duration = cfg["green_duration"]
                            break 
                            
                elif event.operation == "force_mode":
                        mode = json.loads(event.parameters)
                        self._send_diag(f"Режим сменен на {event.parametrs}")
                        break
                
            except Empty:
                sleep(0.2)
                #attempts -= 1
        self._send_diag("Отключение")
        print(f'[{self.__class__.__name__}] завершение работы')

    def send_diagnostics(self, message):
        event = Event(
            source="LightsGPIO",
            destination="SelfDiagnostics",
            operation="diagnostic_log",
            parameters=message
        )
        self.monitor_queue.put(event)

In [ ]:
class SelfDiagnostics(Process):
    def __init__(self, duration=10):
        super().__init__()
        self._own_queue = Queue()
        self.daemon = True
        self.duration = duration

    def entity_queue(self):
        return self._own_queue

    def run(self):
        print(f'[{self.__class__.__name__}] старт')
        start_time = time.time()
        while time.time() - start_time < self.duration:
            try:
                event = self._own_queue.get_nowait()
                if event.operation == "diagnostic_log":
                    print(f"[{self.__class__.__name__}] {event.parameters}")
            except Empty:
                sleep(0.1)
        print(f'[{self.__class__.__name__}] завершение работы')


In [ ]:
class CitySystemConnector(Process):
    def __init__(self, monitor_queue: Queue):
        super().__init__()
        self.monitor_queue = monitor_queue
        self.daemon = True

    def run(self):
        print(f'[{self.__class__.__name__}] Изменение длительности зеленого')
        config_payload = {"green_duration": 10, "yellow_duration": 5}
        event_cfg = Event(
            source="CitySystemConnector",
            destination="ControlSystem",
            operation="update_config",
            parameters=json.dumps(config_payload)
        )
        self.monitor_queue.put(event_cfg)
        print(f'[{self.__class__.__name__}] Принудительный режим "Мигающий желтый"')
        override_payload = {"direction_1": "yellow_blinking", "direction_2": "yellow_blinking"}
        event_override = Event(
            source="CitySystemConnector",
            destination="ControlSystem",
            operation="force_mode",
            parameters=json.dumps(override_payload)
        )
        self.monitor_queue.put(event_override)

### Инициализируем монитор и сущности

In [33]:
monitor = Monitor(monitor_events_queue, duration)
control_system = ControlSystem(monitor_events_queue)
lights_gpio = LightsGPIO(monitor_events_queue, duration)

регистрируем очереди сущностей в мониторе

In [34]:
monitor.add_entity_queue(control_system.__class__.__name__, control_system.entity_queue())
monitor.add_entity_queue(lights_gpio.__class__.__name__, lights_gpio.entity_queue())

[монитор] регистрируем сущность ControlSystem
[монитор] регистрируем сущность LightsGPIO


### Запускаем всё

Ожидаемая последовательность событий

![Диаграмма последовательности вызовов](https://www.plantuml.com/plantuml/png/dPBVIiCm6CNlynIvdtk1NSZ02n4KXJr1w886-cUqcR0xgoBpIdoJLgqhtRg-mfStygJPM3js9OLyoPTpVia97ITQn7eU-6o6gZmr4w7c5r6euyYVB18j0ouIxhb6JpIHtZnMUd4JXKf7iPK5RjgJNQlx1vrStbtTMeNVhXZR0VdmV6yQ34QSQjhI5sNcWrE5QMrUgQHlysAUq7oZqcxyq1hbW6KxG8S5KWEBPHMe5MLeOBa6uHd4YWbVSs1JQg1OKktEF3ATSTI2Vk7Os6b6AupGerctn3uM5WWfn_OAxGRGr4OogTtktjCzmt3utyZ2q-fHQBb_JrSEv177oHigRB1E2ChOL1vxfPz8ZWjdBhv9ELn5Bw_vfFf4L2fFlZsEtkBBpNjxWH8mkyHWbbZbC1TCXbCsne1Vxmy0)

In [35]:
monitor.start()
control_system.start()
lights_gpio.start()
sleep(2)

[монитор] старт
[ControlSystem] старт
[LightsGPIO] старт
[ControlSystem] отправляем тестовый запрос
[ControlSystem] завершение работы
[монитор] обрабатываем событие Event(source='ControlSystem', destination='LightsGPIO', operation='set_mode', parameters='{"direction_1": "green", "direction_2": "green"}')
[монитор] проверяем конфигурацию {'direction_1': 'green', 'direction_2': 'green'}
[монитор] событие не разрешено политиками безопасности


### Теперь останавливаем

In [37]:
monitor.stop()
monitor.join()

[LightsGPIO] завершение работы


## Заключение

В этом блокноте продемонстрирован базовый функционал контролируемого изменения режима работы светофора. 

В примере не реализованы некоторые сущности и большая часть логики работы светофора, которую можно предположить по архитектурной диаграмме. Попробуйте сделать это самостоятельно!

## Упражнения

Уровень "Новичок"

- в коде ControlSystem измените режим на недопустимый (два зелёных) и выполните все ячейки. Убедитесь, что монитор безопасности заблокировал сообщение, как нарушающее политику безопасности
- измените политики безопасности так, чтобы был возможен режим "моргающий жёлтый" (yellow_blinking), переводящий перекрёсток в режим нерегулируемого

Уровень "Средней сложности"

- добавьте политики безопасности для доп. секций со стрелками (поворот налево или направо)
- измените код сущностей, чтобы они не завершали работу после одного сообщения, а работали произвольное время (см. реализацию монитора безопасности)
- реализуйте сущность само-диагностики (SelfDiagnostics) и отправку сообщений от LightsGPIO (необходимо доработать политики безопасности!)

Уровень "Продвинутый"

- измените код сущности ControlSystem, реализуйте смену режимов по таймеру (заданную длительность зелёного по каждому направлению)
- реализуйте сущность CitySystemConnector, которая будет имитировать получение изменения режима, реализуйте взаимодействие CitySystemConnector и ControlSystem (понадобится доработать политики безопасности). Например, изменение длительности зелёного по направлениям или отключение светофора (перевод перекрёстка в режим нерегулируемого)
- реализуйте передачу в компонент CitySystemConnector информации об исправности светофора (статус самодиагностики; текущий режим работы). В компоненте реализуйте вывод в виде сообщений о состоянии системы